# 21 · Validating with Pydantic and retrying
Level L2 · Part 3 · core · ~30 min · builds on 20

## Goal
Write `extract(note, model)`: it validates each reply against a Pydantic class, sends the validation
errors back to the model when a reply is wrong, and gives up after 3 attempts. Run it on a scripted fake
model first, then on Gemini.

## The idea
Lesson 20 ended with a hole: no way of asking checks that the *values* are right. **Pydantic validation**
closes it on our side. We write the rules once in a class (date is `YYYY-MM`, topics are lowercase, the
year really appears in the note), and Pydantic tells us exactly which rule a reply broke. A
**validation retry** sends that message back to the model: "this is what was wrong, try again". A
**retry cap** stops the loop after a few tries. Think of a teacher handing back homework with the
mistakes circled: a second try usually fixes them, but after three tries you stop and escalate.

## Picture
```mermaid
flowchart LR
    P[prompt] --> M[model]
    M --> V{Pydantic<br/>validates?}
    V -- yes --> R[record]
    V -- "no, attempts < 3" --> F["add reply + errors<br/>to the messages"]
    F --> M
    V -- "no, attempt 3" --> X[ExtractionFailed]
```

## Step by step
### 1. Start the recorder and set up
The recorder first, as in lesson 20. Without a key the Gemini calls replay from
`../data/llm_cassettes/21_validate-and-retry.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("21_validate-and-retry")

LLM mode: replay (21_validate-and-retry.yaml)


Setup from lesson 20: the notes, client, model id and the three test notes. Note 5 is the one with a date.

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
TEST_NOTES = [NOTES[0], NOTES[2], NOTES[4]]
print([n["id"] for n in TEST_NOTES], "| model:", MODEL)

[1, 3, 5] | model: gemini-3.5-flash-lite


### 2. Rules in a Pydantic class
Lesson 20's `NoteRecord` only described the shape. Now it also holds the rules. `Field` adds simple
limits (a regex `pattern`, list lengths). A `field_validator` is a method for any rule you can write in
Python; it raises `ValueError` to reject a value.

In [3]:
import re
from pydantic import BaseModel, Field, ValidationError, ValidationInfo, field_validator

class NoteRecord(BaseModel):
    title: str = Field(min_length=1, description="short title, under 8 words")
    topics: list[str] = Field(min_length=1, max_length=3, description="1 to 3 lowercase topic words")
    date: str | None = Field(pattern=r"^\d{4}-\d{2}$", description="month in the note, YYYY-MM, else null")

    @field_validator("topics")
    @classmethod
    def lowercase(cls, topics: list[str]) -> list[str]:
        bad = [t for t in topics if t != t.lower()]
        if bad:
            raise ValueError(f"topics must all be lowercase, {len(bad)} are not")
        return topics

Validate a good reply and a bad one. `model_validate_json` parses the JSON text *and* checks every rule
in one step.

In [4]:
good = '{"title": "Perseverance lands on Mars", "topics": ["mars", "rovers"], "date": "2021-02"}'
bad = '{"title": "Perseverance lands on Mars", "topics": ["Mars", "Rovers"], "date": "February 2021"}'

print(NoteRecord.model_validate_json(good))
try:
    NoteRecord.model_validate_json(bad)
except ValidationError as e:
    print(f"{e.error_count()} errors")
    for err in e.errors():
        print(" ", err["loc"], "|", err["type"], "|", err["msg"])

title='Perseverance lands on Mars' topics=['mars', 'rovers'] date='2021-02'
2 errors
  ('topics',) | value_error | Value error, topics must all be lowercase, 2 are not
  ('date',) | string_pattern_mismatch | String should match pattern '^\d{4}-\d{2}$'


Pydantic checks **every field** and reports all the failing ones in one exception, each with a location
(`loc`), a type and a message: what a model needs to fix its reply in one go. Inside one field the rules
run in order, and a `field_validator` runs only after the `Field` limits pass.

### 3. A rule that checks the value against the note
A schema cannot know the note. A validator can, if we pass the note in through **validation context**:
`model_validate_json(text, context={...})` hands the dict to every validator as `info.context`. This rule
rejects a date whose year is not in the note: a made-up date.

In [5]:
class CheckedRecord(NoteRecord):
    @field_validator("date")
    @classmethod
    def year_in_note(cls, date: str | None, info: ValidationInfo) -> str | None:
        note = (info.context or {}).get("note", "")
        if date and date[:4] not in note:
            raise ValueError(f"year {date[:4]} does not appear in the note")
        return date

made_up = '{"title": "Two moons", "topics": ["mars"], "date": "2019-05"}'
try:
    CheckedRecord.model_validate_json(made_up, context={"note": NOTES[0]["text"]})
except ValidationError as e:
    print(e.errors()[0]["msg"])

Value error, year 2019 does not appear in the note


This is the "values correct" column that every row of lesson 20's table left empty. It catches only what
you write a rule for, but it is checked by code on every reply.

### 4. Turn errors into feedback for the model
`str(e)` works but is noisy: it repeats the whole input and adds a docs URL per error. A short line per
error, with the bad value, is cheaper and clearer.

In [6]:
def explain(e: ValidationError) -> str:
    lines = []
    for err in e.errors(include_url=False):
        where = ".".join(str(part) for part in err["loc"]) or "reply"
        lines.append(f"- {where}: {err['msg']} (got {err['input']!r})")
    return "\n".join(lines)

try:
    CheckedRecord.model_validate_json(bad, context={"note": NOTES[4]["text"]})
except ValidationError as e:
    print(f"str(e) is {len(str(e))} characters; explain(e) is {len(explain(e))}:")
    print(explain(e))

str(e) is 468 characters; explain(e) is 161:
- topics: Value error, topics must all be lowercase, 2 are not (got ['Mars', 'Rovers'])
- date: String should match pattern '^\d{4}-\d{2}$' (got 'February 2021')


### 5. A scripted fake model
Before spending real calls, a fake model plays back fixed replies and remembers every message list it
was sent. A model here is any function `messages -> text`, with messages as `{"role", "text"}` dicts:
the chat-history format from lesson 06, which lesson 08 maps to Gemini.

In [7]:
class ScriptedModel:
    def __init__(self, replies: list[str]):
        self.replies = list(replies)
        self.seen = []                        # every message list we were called with

    def __call__(self, messages: list[dict]) -> str:
        self.seen.append([dict(m) for m in messages])
        return self.replies.pop(0)

fake = ScriptedModel([bad, good])
print(fake([{"role": "user", "text": "hi"}])[:40], "...")
print("calls so far:", len(fake.seen))

{"title": "Perseverance lands on Mars",  ...
calls so far: 1


### 6. The retry loop with a cap
The prompt is lesson 20's casual one. On a failure, the loop adds the model's own bad reply and a user
message with the errors, so the model sees what it said and what was wrong. After `max_attempts` it
raises. The rules class is a parameter, `schema`, so you can pass a stricter one later.

In [8]:
class ExtractionFailed(Exception):
    pass

def casual_prompt(note: dict) -> str:
    return f"Extract the title, topics and date from this note as JSON: {note['text']}"

def extract(note: dict, model, schema: type[BaseModel] = CheckedRecord,
            max_attempts: int = 3) -> tuple[BaseModel, int]:
    messages = [{"role": "user", "text": casual_prompt(note)}]
    for attempt in range(1, max_attempts + 1):
        reply = model(messages)
        try:
            record = schema.model_validate_json(reply, context={"note": note["text"]})
            print(f"  attempt {attempt}: valid")
            return record, attempt
        except ValidationError as e:
            print(f"  attempt {attempt}: {e.error_count()} error(s)\n{explain(e)}")
            messages += [{"role": "assistant", "text": reply},
                         {"role": "user", "text": f"Your JSON failed validation:\n{explain(e)}\n"
                                                  "Reply again with the corrected JSON only."}]
    raise ExtractionFailed(f"note {note['id']}: no valid record after {max_attempts} attempts")

Run it with a fresh fake: a bad reply first, then a good one.

In [9]:
fake = ScriptedModel([bad, good])
record, attempts = extract(NOTES[4], fake)
print(record, "| attempts:", attempts)

  attempt 1: 2 error(s)
- topics: Value error, topics must all be lowercase, 2 are not (got ['Mars', 'Rovers'])
- date: String should match pattern '^\d{4}-\d{2}$' (got 'February 2021')
  attempt 2: valid
title='Perseverance lands on Mars' topics=['mars', 'rovers'] date='2021-02' | attempts: 2


The fake remembered what it was sent. The second call carried three messages: the prompt, the bad reply,
and the error list.

In [10]:
for m in fake.seen[1]:
    print(f"[{m['role']}] {m['text']}\n")

[user] Extract the title, topics and date from this note as JSON: The rover Perseverance landed on Mars in February 2021.

[assistant] {"title": "Perseverance lands on Mars", "topics": ["Mars", "Rovers"], "date": "February 2021"}

[user] Your JSON failed validation:
- topics: Value error, topics must all be lowercase, 2 are not (got ['Mars', 'Rovers'])
- date: String should match pattern '^\d{4}-\d{2}$' (got 'February 2021')
Reply again with the corrected JSON only.



### 7. The cap in action
A model that never gets it right. The loop must stop, not spin forever and spend money.

In [11]:
hopeless = ScriptedModel([bad] * 10)
try:
    extract(NOTES[4], hopeless)
except ExtractionFailed as e:
    print("ExtractionFailed:", e)
print("calls made:", len(hopeless.seen), "| messages in each call:", [len(s) for s in hopeless.seen])

  attempt 1: 2 error(s)
- topics: Value error, topics must all be lowercase, 2 are not (got ['Mars', 'Rovers'])
- date: String should match pattern '^\d{4}-\d{2}$' (got 'February 2021')
  attempt 2: 2 error(s)
- topics: Value error, topics must all be lowercase, 2 are not (got ['Mars', 'Rovers'])
- date: String should match pattern '^\d{4}-\d{2}$' (got 'February 2021')
  attempt 3: 2 error(s)
- topics: Value error, topics must all be lowercase, 2 are not (got ['Mars', 'Rovers'])
- date: String should match pattern '^\d{4}-\d{2}$' (got 'February 2021')
ExtractionFailed: note 5: no valid record after 3 attempts
calls made: 3 | messages in each call: [1, 3, 5]


Three calls, then a clear exception the caller can handle (skip the note, log it, ask a person). Each
retry also resends the whole conversation, so the calls get longer: 1, 3, then 5 messages. Each retry
resends a longer conversation, so cost grows with every attempt; a small cap (2 or 3 retries) bounds it.

### 8. The same loop on Gemini
An adapter turns our messages into Gemini `Content` (role `assistant` becomes `model`, as in lesson 08)
and returns the text. We use **JSON mode only, no schema**: in lesson 20 that combination returned
Title Case topics and `"February 2021"`, so there is something for validation to catch.

In [12]:
JSON_MODE = types.GenerateContentConfig(max_output_tokens=200, temperature=0, automatic_function_calling=NO_AFC,
                                        response_mime_type="application/json")
real_calls = 0

def gemini(messages: list[dict]) -> str:
    global real_calls
    real_calls += 1
    contents = [types.Content(role="model" if m["role"] == "assistant" else "user",
                              parts=[types.Part(text=m["text"])]) for m in messages]
    resp = client.models.generate_content(model=MODEL, contents=contents, config=JSON_MODE)
    return resp.text or ""

Same `extract`, same rules, a real model. Up to 3 calls per note, 9 at most.

In [13]:
real_results = {}
for n in TEST_NOTES:
    print(f"note {n['id']}:")
    try:
        real_results[n["id"]] = extract(n, gemini)
    except ExtractionFailed as e:
        print("  gave up:", e)
print("real calls:", real_calls)

note 1:
  attempt 1: 2 error(s)
- title: Input should be a valid string (got None)
- topics: List should have at most 3 items after validation, not 4 (got ['Mars', 'Moons', 'Phobos', 'Deimos'])
  attempt 2: 1 error(s)
- topics: Value error, topics must all be lowercase, 3 are not (got ['Mars', 'Phobos', 'Deimos'])
  attempt 3: valid
note 3:
  attempt 1: 1 error(s)
- topics: Value error, topics must all be lowercase, 3 are not (got ['Mars', 'Astronomy', 'Time'])
  attempt 2: valid
note 5:
  attempt 1: 2 error(s)
- topics: Value error, topics must all be lowercase, 3 are not (got ['Rover Perseverance', 'Mars', 'Space exploration'])
- date: String should match pattern '^\d{4}-\d{2}$' (got 'February 2021')
  attempt 2: valid
real calls: 7


The records Scout can now store, and how many attempts each needed.

In [14]:
for note_id, (rec, attempts) in real_results.items():
    print(f"note {note_id} ({attempts} attempt(s), first reply valid: {attempts == 1}): {rec.model_dump()}")

note 1 (3 attempt(s), first reply valid: False): {'title': 'Mars Moons', 'topics': ['mars', 'phobos', 'deimos'], 'date': None}
note 3 (2 attempt(s), first reply valid: False): {'title': 'A day on Mars', 'topics': ['mars', 'astronomy', 'time'], 'date': None}
note 5 (2 attempt(s), first reply valid: False): {'title': 'Rover Perseverance Landing', 'topics': ['rover perseverance', 'mars', 'space exploration'], 'date': '2021-02'}


All three notes ended with a valid record, in 7 calls instead of 3. The prompt never stated a rule, and
no first reply was valid (`first reply valid: False` above): the model learned the rules only from the
feedback. Note 1 needed a third try: its first reply had 4 topics, so the lowercase validator never ran
(step 2), and that error only showed up on attempt 2.

Validation fixes what it checks: note 5's `"February 2021"` became `"2021-02"`. It catches only what you
wrote a rule for: `"rover perseverance"` is two words, yet it passed. In real use, put the rules in the
prompt or schema too (lesson 20) so most first replies pass, and keep the retry as the backstop.

## At work
- **Schema first, validation as the backstop.** Teams use a schema where the API has one, still validate
  every reply with Pydantic, retry with the errors fed back, cap at 2 or 3, and log the failure rate.
  → lesson 48 (evals), lesson 49 (logging and tracing)
- **The same loop runs on tool arguments.** Tool arguments are validated with Pydantic even under
  `strict` schemas; Anthropic notes Claude retries an invalid tool call 2 or 3 times before giving up.
  → lesson 24 (strict schemas), lesson 28 (recoverable tool errors)
- **Validation retries are not network retries.** A 429 or 503 is retried by the transport layer
  (lesson 14) and never shown to the model; only a bad *answer* goes back as feedback. Retrying a step
  with a side effect is its own problem. → lesson 39
- **Libraries package this loop.** Instructor and PydanticAI take a Pydantic class and `max_retries` and
  do what `extract` does. → track 3 (frameworks)

## What just happened
- Rules went into a Pydantic class: `Field` limits, a `field_validator`, and a validator that checks the
  date against the note through validation context.
- `model_validate_json` parsed and checked in one step and reported every broken rule in one
  `ValidationError`; `explain` turned that into short feedback.
- `extract` sent the model's bad reply and the errors back, so the fake model's second reply passed.
- The cap stopped a hopeless model after 3 calls with `ExtractionFailed`; each retry resent a longer
  conversation.
- The same loop ran unchanged on Gemini, which only needed a small adapter.

## Common mistakes
**1. Retrying without the feedback.** Sending the same prompt again just asks the same question. This
fake model, like a real one, fixes its reply only when it is told what was wrong.

In [15]:
def stubborn(messages: list[dict]) -> str:
    return good if "failed validation" in messages[-1]["text"] else bad

def blind_extract(note: dict, model, max_attempts: int = 3) -> CheckedRecord:
    for attempt in range(max_attempts):
        try:
            return CheckedRecord.model_validate_json(model([{"role": "user", "text": casual_prompt(note)}]),
                                                     context={"note": note["text"]})
        except ValidationError:
            pass                                  # error thrown away, nothing learned
    raise ExtractionFailed(f"note {note['id']}: no valid record after {max_attempts} attempts")

blind_extract(NOTES[4], stubborn)

ExtractionFailed: note 5: no valid record after 3 attempts

Fix: feed the errors back, as `extract` does. The same model now succeeds on the second attempt.

In [16]:
record, attempts = extract(NOTES[4], stubborn)
print("attempts:", attempts)

  attempt 1: 2 error(s)
- topics: Value error, topics must all be lowercase, 2 are not (got ['Mars', 'Rovers'])
- date: String should match pattern '^\d{4}-\d{2}$' (got 'February 2021')
  attempt 2: valid
attempts: 2


**2. Calling `model_validate` on JSON text.** `model_validate` expects a dict or an object; a string from
the model is neither.

In [17]:
CheckedRecord.model_validate(good)

ValidationError: 1 validation error for CheckedRecord
  Input should be a valid dictionary or instance of CheckedRecord [type=model_type, input_value='{"title": "Perseverance ...s"], "date": "2021-02"}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/model_type

Fix: use `model_validate_json` for text (it parses and validates in one step), or `json.loads` first.

In [18]:
print(CheckedRecord.model_validate_json(good, context={"note": NOTES[4]["text"]}))

title='Perseverance lands on Mars' topics=['mars', 'rovers'] date='2021-02'


## Try it
Subclass `CheckedRecord` with a rule that the title has at most 6 words. Then build a `ScriptedModel`
whose first reply has a 10-word title (everything else valid for note 1) and whose second is fixed, and
run `extract(NOTES[0], fake, schema=YourClass)`. It should succeed on attempt 2, and the feedback should
name `title`.

In [19]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [20]:
llm_replay.stop()

## Key terms
- **Pydantic validation** — checking data against a Pydantic class's types and rules; failures raise one `ValidationError` listing every broken rule.
- **ValidationError** — Pydantic's exception; `.errors()` gives one dict per problem with `loc`, `type`, `msg` and `input`.
- **field validator** — a `@field_validator` method on a Pydantic class that checks or changes one field and raises `ValueError` to reject it.
- **validation context** — extra data passed with `model_validate_json(..., context=...)` that validators read as `info.context`, e.g. the source note.
- **validation retry** — sending the model its invalid reply plus the validation errors and asking again.
- **retry cap** — the maximum number of attempts before the loop gives up and raises, bounding cost and time.